# ROMS-DART Data Assimilation Experiment: STATE-SPACE

This notebook goes over the EnKF output from DART and diagnoses the results in state-space. 
The converter itself should be run from the terminal. This notebook focuses on reading, summarizing, and visualizing the assimilation results.

## Objectives

In this notebook you will learn how to:

- Read ROMS prior and posterior ensemble files
- Visualize assimilation increments in the ocean state
- Compare prior and posterior analyses
- Monitor the spatial reduction of ensemble spread
- Assess adaptive inflation patterns 

## 1. Import Python libraries

In [ ]:
import pydartdiags.obs_sequence.obs_sequence as obsq

import os
import cmocean
import numpy             as np
import pandas            as pd
import xarray            as xr
import plotly.io         as pio
import cartopy.crs       as ccrs
import plotly.express    as px
import cartopy.feature   as cfeature
import matplotlib.pyplot as plt
import matplotlib.dates  as mdates

from pathlib              import Path
from IPython.display      import Markdown, display
from matplotlib.colors    import LogNorm
from pydartdiags.stats    import stats
from pydartdiags.matplots import matplots as mp

## 2. Define Paths and Variables

In [ ]:
# Path to DART repo (directory) 
basedir = Path(f"/glade/derecho/scratch/{os.environ['USER']}/inacawo/DART_training/models/ROMS_rutgers/work")

# Path to the obs_seq and obs_diag_output.nc files
mohadir = Path(f"/glade/derecho/scratch/gharamti/inacawo/DART_training/models/ROMS_rutgers/work")

# Roms restart template 
template = basedir / 'roms_template.nc'

run_dirs = {
    'exp1': basedir / 'filter_output'       ,  # hloc: 300km, vloc: 250m
    # 'exp2': mohadir / 'filter_output_300_50', 
    # 'exp3': mohadir / 'filter_output_300_no', 
    # 'exp4': mohadir / 'filter_output_eval'
}

files = {
    'increment' : 'increment.nc'    ,
    'prior_mean': 'preassim_mean.nc', 
    'post_mean' : 'output_mean.nc'  , 
    'prior_sd'  : 'preassim_sd.nc'  ,
    'post_sd'   : 'output_sd.nc'    , 
    'inflate'   : 'output_priorinf_mean.nc'
}

state = {
    'T':   {'name': 'temp', 'grid': 'rho', 'unit': 'C'},
    'S':   {'name': 'salt', 'grid': 'rho', 'unit': 'psu'},
    'SSH': {'name': 'zeta', 'grid': 'rho', 'unit': 'm'},
    'U':   {'name': 'u',    'grid': 'u',   'unit': 'm/s'},   
    'V':   {'name': 'v',    'grid': 'v',   'unit': 'm/s'}
}

field     = ['T', 'S', 'U', 'V', 'SSH']
Variables = ['Temperature', 'Salinity', 'U Current', 'V Current', 'Sea Surface Height'] 

## 3. DA Increments

In [ ]:
# Functions & Tools
datasets = {
    exp: {
        file_key: xr.open_dataset(path / fname)
        for file_key, fname in files.items()
    }
    for exp, path in run_dirs.items()
}

def get_2D_field(var_key, ds, roms, level):
    vname = state[var_key]['name']
    grid  = state[var_key]['grid']

    if vname == 'zeta':
        data = ds[vname].squeeze()
    else:
        data = ds[vname].isel(s_rho=level).squeeze()

    if grid == 'rho':
        lons = roms['lon_rho']
        lats = roms['lat_rho']

    elif grid == 'u':
        lons = roms['lon_u']
        lats = roms['lat_u']

    elif grid == 'v':
        lons = roms['lon_v']
        lats = roms['lat_v']

    return data, lons, lats
    
def uv_to_rho(u, v):

    urho = 0.5 * (u[:, :-1] + u[:, 1:])
    vrho = 0.5 * (v[:-1, :] + v[1:, :])

    urho = np.pad(urho, ((0,0),(1,1)), mode='edge')
    vrho = np.pad(vrho, ((1,1),(0,0)), mode='edge')

    return urho, vrho

def roms_z_section(ds, eta):
    h    = ds["h"].isel(eta_rho=eta).values
    zeta = ds["zeta"].isel(eta_rho=eta).squeeze().values

    s_rho = ds["s_rho"].values[:, None]
    Cs_r  = ds["Cs_r"].values[:, None]
    hc    = float(ds["hc"].values)

    z0 = (hc * s_rho + h[None, :] * Cs_r) / (hc + h[None, :])
    z  = zeta[None, :] + (zeta[None, :] + h[None, :]) * z0

    return -z

def roms_z_section_meridional(ds, xi):
    h    = ds["h"].isel(xi_rho=xi).values
    zeta = ds["zeta"].isel(xi_rho=xi).squeeze().values

    s_rho = ds["s_rho"].values[:, None]
    Cs_r  = ds["Cs_r"].values[:, None]
    hc    = float(ds["hc"].values)

    z0 = (hc * s_rho + h[None, :] * Cs_r) / (hc + h[None, :])
    z  = zeta[None, :] + (zeta[None, :] + h[None, :]) * z0

    return -z

display(Markdown(
    """### State-Space Diagnostics

Unlike observation-space diagnostics, which evaluate the fit between the model and observations, 
state-space diagnostics examine how the assimilation modifies the model state itself. 
These diagnostics help answer several important questions:

* Where did the observations influence the model?
* Which state variables experienced the largest adjustments?
* How did the analysis modify ocean circulation and sea surface height?
* Did the assimilation reduce ensemble uncertainty?
* How much inflation was applied to maintain ensemble spread?
"""))

In [ ]:
# Compare T/S increments at different levels
run   = 'exp1'
stage = 'increment'

roms  = xr.open_dataset(template)
lay   = 1
lev   = 70-lay 

m_col = 'RdBu_r'

Ti, lon, lat = get_2D_field(field[0], datasets[run][stage], roms, lev)
Si,  _ ,  _  = get_2D_field(field[1], datasets[run][stage], roms, lev)

min_T, min_S = np.nanmin(Ti), np.nanmin(Si)
max_T, max_S = np.nanmax(Ti), np.nanmax(Si)

camp_T = np.minimum(abs(min_T), abs(max_T))
camp_S = np.minimum(abs(min_S), abs(max_S))

proj = ccrs.PlateCarree()

fig, axes = plt.subplots(
    1, 2,
    figsize=(16,8),
    subplot_kw=dict(projection=proj)
)

pcm = axes[0].pcolormesh(
    lon, lat, Ti,
    cmap=m_col,
    vmin=-camp_T,
    vmax=+camp_T,
    transform=proj
)

axes[1].pcolormesh(
    lon, lat, Si,
    cmap=m_col,
    vmin=-camp_S,
    vmax=+camp_S,
    transform=proj
)

for k, ax in enumerate(axes):
    ax.set_facecolor('aliceblue') 
    ax.add_feature(cfeature.LAND, facecolor='whitesmoke', zorder=1) 
    ax.add_feature(cfeature.NaturalEarthFeature('physical', 'coastline', '10m'), edgecolor='black', facecolor='none', zorder=2)
    ax.add_feature(cfeature.LAKES, facecolor='lightsteelblue', zorder=2)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, zorder=2)

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.1)
    gl.top_labels = False
    gl.right_labels = False

    cbar = fig.colorbar(pcm, ax=ax, shrink=0.4)
    cbar.set_label(state[field[k]]['unit'])

    ax.set_title(f"{Variables[k]}, Layer: {lay}", fontsize= 16)

plt.tight_layout()

display(Markdown(
    """### Temperature and Salinity Increments

The increment is defined as the difference between the posterior and prior ensemble means.

Positive increments indicate locations where the assimilation increased the state variable, 
while negative increments indicate locations where it decreased the state variable. 
Examining increments helps identify where observational information is being introduced 
into the model and how strongly different regions are affected.
"""))

plt.show()

In [ ]:
# SSH increments at the surface
run   = 'exp1'
stage = 'increment'
lay   = 1
lev   = 70-lay 

k = 4

m_col = 'RdBu_r'

Zi, lon, lat = get_2D_field(field[k], datasets[run][stage], roms, lev)

min_Z, max_Z = np.nanmin(Zi), np.nanmax(Zi)

camp_Z = np.minimum(abs(min_Z), abs(max_Z))

proj = ccrs.PlateCarree()

fig = plt.figure(figsize=(12, 8))
ax  = plt.axes(projection=proj)

pcm = ax.pcolormesh(lon, lat, Zi,
                    cmap = m_col,
                    transform=proj)

pcm = ax.pcolormesh(
    lon, lat, Zi,
    cmap=m_col,
    vmin=-camp_Z,
    vmax=+camp_Z,
    transform=proj
)

ax.set_facecolor('aliceblue') 
ax.add_feature(cfeature.LAND, facecolor='whitesmoke') 
ax.add_feature(cfeature.NaturalEarthFeature('physical', 'coastline', '10m'), edgecolor='black', facecolor='none')
ax.add_feature(cfeature.LAKES, facecolor='lightsteelblue')
ax.add_feature(cfeature.BORDERS, linewidth=0.5)

gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(pcm, shrink=0.5, pad= 0.03)
cbar.set_label(state[field[k]]['unit'])

plt.title(f"{Variables[k]} Increments", fontsize= 16)

display(Markdown(
    """### Sea Surface Height Increments

Sea surface height (SSH) increments provide a direct view of how assimilated observations modify the ocean surface state. 
Because SSH is dynamically linked to ocean circulation, adjustments to SSH often indicate corresponding changes in currents and subsurface structure.
Spatially coherent increment patterns generally suggest that the assimilation is producing physically consistent updates rather than isolated point corrections.
"""))
plt.show()

In [ ]:
# SSH + currents
run    = 'exp1' 
lay    = 1
lev    = 70-lay 
skip   = 50
arrows = 14
stage  = ['prior_mean', 'post_mean']
col_m  = 'bwr'

U_f, *_ = get_2D_field(field[2], datasets[run][stage[0]], roms, lev)
V_f, *_ = get_2D_field(field[3], datasets[run][stage[0]], roms, lev)

U_a, *_ = get_2D_field(field[2], datasets[run][stage[1]], roms, lev)
V_a, *_ = get_2D_field(field[3], datasets[run][stage[1]], roms, lev)

U_f_rho, V_f_rho = uv_to_rho(U_f, V_f)
U_a_rho, V_a_rho = uv_to_rho(U_a, V_a) 

lon_q = lon[::skip, ::skip]
lat_q = lat[::skip, ::skip]

u_pre_q  = U_f_rho[::skip, ::skip]
v_pre_q  = V_f_rho[::skip, ::skip]
u_post_q = U_a_rho[::skip, ::skip]
v_post_q = V_a_rho[::skip, ::skip]

SSH_f, _, _ = get_2D_field(field[4], datasets[run][stage[0]], roms, lev)
SSH_a, _, _ = get_2D_field(field[4], datasets[run][stage[1]], roms, lev)

min_SSH_f, min_SSH_a = np.nanmin(SSH_f), np.nanmin(SSH_a)
max_SSH_f, max_SSH_a = np.nanmax(SSH_f), np.nanmax(SSH_a)

camp_f = np.minimum(abs(min_SSH_f), abs(max_SSH_f))
camp_a = np.minimum(abs(min_SSH_a), abs(max_SSH_a))

fig, axes = plt.subplots(
    1, 2,
    figsize=(18,6),
    subplot_kw=dict(projection=proj)
)

pcm = axes[0].pcolormesh(lon, lat, SSH_f,
                    vmin=-camp_f,
                    vmax=+camp_f,
                    cmap=col_m,
                    transform=proj)

axes[0].quiver(
    lon_q, lat_q,
    u_pre_q, v_pre_q,
    transform=proj,
    scale=arrows
)

cbar = fig.colorbar(pcm, ax=axes[0], shrink=0.5)
cbar.set_label(state[field[4]]['unit'])

pcm = axes[1].pcolormesh(lon, lat, SSH_a,
                    vmin=-camp_a,
                    vmax=+camp_a,
                    cmap=col_m,
                    transform=proj)

axes[1].quiver(
    lon_q, lat_q,
    u_post_q, v_post_q,
    transform=proj,
    scale=arrows
)

cbar = fig.colorbar(pcm, ax=axes[1], shrink=0.5)
cbar.set_label(state[field[4]]['unit'])

for k, ax in enumerate(axes):
    ax.set_facecolor('aliceblue') 
    ax.add_feature(cfeature.LAND, facecolor='whitesmoke') 
    ax.add_feature(cfeature.NaturalEarthFeature('physical', 'coastline', '10m'), edgecolor='black', facecolor='none')
    ax.add_feature(cfeature.LAKES, facecolor='lightsteelblue')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    
    ax.set_title(f'{field[4]}: {stage[k]}', fontsize= 16)

    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.1)
    gl.top_labels   = False
    gl.right_labels = False

plt.tight_layout()

display(Markdown("""
### Prior and Posterior Ocean State

The panels below compare the prior and posterior ensemble means. 
While increment plots show the adjustment applied by the assimilation, 
prior/posterior comparisons reveal how the ocean state itself changes. 
Improvements are often most apparent in regions directly constrained by 
observations or strongly connected through ensemble covariances.
"""))

# plt.savefig('ssh_currents.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Increments in a Vertical Section 
run   = 'exp1'
stage = 'increment'
xnum  = [1, 0]
buff  = 5

dep = roms['s_rho']

float_num = 2
float_lon = [113.89, 131.59]
float_lat = [-5.74, -4.95]
float_lab = ['ARVOR Float A', 'ARVOR Float B']

# Float locations
ref_eta = []
for f in range(float_num):
    dist2 = (lon - float_lon[f])**2 + (lat - float_lat[f])**2
    idx   = np.unravel_index(np.nanargmin(dist2.values), dist2.shape)

    ref_eta.append(idx[0])

fig = plt.figure(figsize=(10, 5))
ax  = plt.axes() 

# Zonal Section (east-west)
f = 1

vname = state[field[xnum[f]]]['name']

X = datasets[run][stage][vname].isel(eta_rho=ref_eta[k]).squeeze().values

# Actual depths along the section (nz x nx)
depth = roms_z_section(roms, ref_eta[k])   

# Longitude along section
lat_s = np.nanmean(roms['lat_rho'].isel(eta_rho=ref_eta[k]).values)
lon_d = roms['lon_rho'].isel(eta_rho=ref_eta[k]).values

lon_d = np.asarray(lon_d)
depth = np.asarray(depth)
X     = np.asarray(X)

valid_col = np.all(np.isfinite(depth), axis=0) & np.any(np.isfinite(X), axis=0)

lon_p   = lon_d[valid_col]
depth_p = depth[:, valid_col]
X_p     = X[:, valid_col]

lim = 3
pcm = ax.pcolormesh(lon_p, depth_p, np.ma.masked_invalid(X_p),
        cmap=cmocean.cm.tarn,
        vmin=-lim, vmax= lim,
        shading="auto")

ax.axvline(float_lon[k], color='tab:red', linestyle='--', lw=2, label=float_lab[k])

ax.set_ylim(500, 0)
ax.set_xlim(np.ceil(float_lon[k] - buff), np.ceil(float_lon[k] + buff))
ax.set_xlabel("Longitude")
ax.set_ylabel("Depth (m)")

ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.4)
ax.legend(loc='lower right', frameon=False, fontsize=12)

ax.set_title(f"{Variables[xnum[k]]} zonal section, Latitude: {lat_s:.2f} N", fontsize=14)
plt.colorbar(pcm, ax=ax, label=f"Increment ({state[field[xnum[k]]]['unit']})", shrink=0.8, pad= 0.02)

float_lon_model = float(lon.isel(eta_rho=ref_eta[k], xi_rho=ref_xi[k]))

ax.axvline(
    float_lon_model,
    color='tab:red',
    linestyle='--',
    lw=2,
    label=float_lab[k]
)
        
plt.tight_layout()

# plt.savefig('temp_inc_zonal_inc.png', dpi=300, bbox_inches='tight')
plt.show()

display(Markdown(
    """### Horizontal Localization Footprint

The zonal section through ARVOR Float B shows that the dominant temperature increments are largely confined to the expected horizontal localization support. 
For this experiment, the localization cutoff corresponds to a Gaspari-Cohn half-width of approximately 150 km and a 0-weight distance of approximately 300 km.
At the float latitude (~5°S), one degree longitude corresponds to approximately: 110.8 km = 111.2 × cos(5°)

Therefore: 300 km ~2.7°. With the float located at approximately 131.6°E, the expected localization support extends from: 128.9°E to 134.3°E

The observed increment pattern is largely confined within this range, indicating that the horizontal localization scale is clearly reflected in the analysis increments.
"""))

In [ ]:
# Spread Reduction 
run   = 'exp1'
stage = ['prior_sd', 'post_sd']
num   = [1, 4] 
lay   = 1
lev   = 70-lay 

m_col = cmocean.cm.matter

sd_f, *_  = get_2D_field(field[num[0]], datasets[run][stage[0]], roms, lev)
sd_a, *_  = get_2D_field(field[num[0]], datasets[run][stage[1]], roms, lev)
X_sd_diff = (sd_f - sd_a) / sd_f * 100

sd_f, *_  = get_2D_field(field[num[1]], datasets[run][stage[0]], roms, lev)
sd_a, *_  = get_2D_field(field[num[1]], datasets[run][stage[1]], roms, lev)
Z_sd_diff = (sd_f - sd_a) / sd_f * 100

fig, axes = plt.subplots(
    1, 2,
    figsize=(18,6),
    subplot_kw=dict(projection=proj)
)

pcm = axes[0].pcolormesh(lon, lat, X_sd_diff, cmap = m_col, transform=proj)
pcm = axes[1].pcolormesh(lon, lat, Z_sd_diff, cmap = m_col, transform=proj)

for k, ax in enumerate(axes):
    ax.set_facecolor('aliceblue') 
    ax.add_feature(cfeature.LAND, facecolor='whitesmoke') 
    ax.add_feature(cfeature.NaturalEarthFeature('physical', 'coastline', '10m'), edgecolor='black', facecolor='none')
    ax.add_feature(cfeature.LAKES, facecolor='lightsteelblue')
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    
    gl = ax.gridlines(draw_labels=True,
                      linewidth=0.5,
                      color='gray',
                      alpha=0.3,
                      linestyle='--')
    
    gl.top_labels = False
    gl.right_labels = False
    
    cb = plt.colorbar(pcm, ax=ax, shrink=0.6)
    cb.set_label('%', fontsize= 14)

    if k == 0:
        ax.set_title(f'{Variables[num[k]]}: Ensemble Spread Reduction at layer {lay}', fontsize= 16)
    elif k == 1:
        ax.set_title(f'{Variables[num[k]]}: Ensemble Spread Reduction', fontsize= 16)      

plt.tight_layout()

display(Markdown("""
### Ensemble Spread Reduction

One of the primary goals of data assimilation is to reduce uncertainty in the model state. 
The ensemble spread provides an estimate of that uncertainty. Comparing prior and posterior spread reveals where observations have increased confidence in the analysis.
Regions with substantial spread reduction indicate locations where observations provided significant information to constrain the model state (E.g., SSH satellite tracks).
"""))

plt.show()

In [ ]:
# Inflation 
run   = 'exp1'
stage = 'inflate'
num   = [0, 4] 
lay   = 1
lev   = 70-lay

m_col = cmocean.cm.delta_r #'turbo'

X_inf, *_ = get_2D_field(field[num[0]], datasets[run][stage], roms, lev)
Z_inf, *_ = get_2D_field(field[num[1]], datasets[run][stage], roms, lev)

fig, axes = plt.subplots(
    1, 2,
    figsize=(18,6),
    subplot_kw=dict(projection=proj)
)

pcm = axes[0].pcolormesh(lon, lat, X_inf, norm=LogNorm(vmin=1e-1, vmax=10), cmap = m_col, transform=proj, zorder= 0)
pcm = axes[1].pcolormesh(lon, lat, Z_inf, norm=LogNorm(vmin=1e-1, vmax=10), cmap = m_col, transform=proj, zorder= 0)

for k, ax in enumerate(axes):
    ax.set_facecolor('aliceblue') 
    ax.add_feature(cfeature.LAND, facecolor='whitesmoke', zorder= 1) 
    ax.add_feature(cfeature.NaturalEarthFeature('physical', 'coastline', '10m'), edgecolor='black', facecolor='none', zorder= 1)
    ax.add_feature(cfeature.LAKES, facecolor='lightsteelblue', zorder= 1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, zorder= 1)
    
    gl = ax.gridlines(draw_labels=True,
                      linewidth=0.5,
                      color='gray',
                      alpha=0.3,
                      linestyle='--')
    
    gl.top_labels = False
    gl.right_labels = False

    if k == 0:
        ax.set_title(f'{Variables[num[k]]}: Inflation at layer {lay}', fontsize= 16)
    elif k == 1:
        ax.set_title(f'{Variables[num[k]]}: Inflation', fontsize= 16)

# ax.set_extent([100, 120, -12, 12], crs=proj)

    cb = plt.colorbar(pcm, ax=ax, shrink=0.5)
    cb.set_label('Inflation')
    cb.set_ticks([0.1, 0.5, 1, 2, 5, 10])
    cb.set_ticklabels(['0.1', '0.5', '1', '2', '5', '10'])

display(Markdown("""
### Inflation Diagnostics

Assimilation naturally reduces ensemble spread. Without additional measures, repeated assimilation cycles can lead 
to an ensemble that becomes unrealistically confident and under-dispersive. Adaptive inflation compensates for 
this tendency by increasing ensemble variance where needed. Examining the inflation field helps identify 
regions where the ensemble required additional variance to maintain consistency with the observing system.
"""))

plt.show()
roms.close()

### Key Takeaways

- The assimilation produces spatially coherent updates to temperature, salinity, SSH, and ocean circulation.
- The largest increments occur in regions directly constrained by observations (SST, SSH) or strongly connected through ensemble covariances.
- Prior-to-posterior comparisons demonstrate that the assimilation modifies both the ocean surface state and circulation patterns.
- Ensemble spread is reduced following assimilation, indicating increased confidence in the analysis.
- Adaptive inflation restores variance where needed and helps maintain a statistically consistent ensemble.
- Together, these diagnostics demonstrate that the ROMS-DART system is successfully updating the ocean state while preserving physically realistic structures.